# AlphaDent — Validation Mask mAP (V13 tiled vs V6 baseline, same metric)

This notebook is **evaluation only**. It answers the one question the V13 tiled-val number
during training could *not*: does V13 (tile-based training) beat the V6 full-image baseline on
the **comparable** metric?

Both checkpoints are scored on the **full validation images** with the **same** self-contained
mask-mAP code (mask-IoU matching -> 10 IoU thresholds 0.5:0.95 -> 101-point interpolated AP), so
the **V13-vs-V6 delta is a true signal**, not a metric artifact.

- **V13** (`V13_best.pt`): tiled + merged inference (`TILE_SIZE`/`OVERLAP` must match `src/01`).
- **V6** (`V6_best.pt`): native full-image inference at `imgsz=768`.
- mAP uses a low confidence (`EVAL_CONF ~ 0.001`) so the precision/recall curve is not truncated.

The historical baseline is **~0.234** Mask mAP50-95 (V6/V10). The absolute number here may differ
slightly from the native Ultralytics `val` (different mask resolution), but it is identical in
treatment for both models, so the comparison is fair. **Decision rule:** V13 only wins if it
clears V6 by more than the project's ~0.003 noise floor.

**Setup:** add two Kaggle Datasets containing the files `V6_best.pt` and `V13_best.pt`, plus the
competition dataset (for the validation images + labels).

## 1. Install Ultralytics

If the notebook has Internet enabled this just installs from PyPI. For an offline notebook, add an
ultralytics wheel dataset and the cell will fall back to it.

In [ ]:
import os
import sys
import subprocess
import importlib.util
from pathlib import Path

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

ALLOW_INTERNET_INSTALL = True   # set False for an offline submission-style environment


def _package_available(name):
    return importlib.util.find_spec(name) is not None


def _find_wheel_dirs(root=Path("/kaggle/input")):
    if not root.exists():
        return []
    wheels = sorted(root.rglob("*.whl"))
    return sorted({p.parent for p in wheels})


if not _package_available("ultralytics"):
    wheel_dirs = _find_wheel_dirs()
    if wheel_dirs:
        cmd = [sys.executable, "-m", "pip", "install", "--no-index"]
        for d in wheel_dirs:
            cmd += ["--find-links", str(d)]
        cmd.append("ultralytics")
        subprocess.check_call(cmd)
    elif ALLOW_INTERNET_INSTALL:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ultralytics"])
    else:
        raise RuntimeError("ultralytics not installed and no offline wheels found.")
else:
    print("ultralytics already installed.")

## 2. Imports and Configuration

`TILE_SIZE` and `OVERLAP` **must match** the values used to train V13 in `src/01`.
`EVAL_CONF` is intentionally low (mAP needs the full PR curve) and is unrelated to the submission
confidence threshold.

In [ ]:
import gc
import numpy as np
import pandas as pd
import cv2
import torch
from tqdm.auto import tqdm
from ultralytics import YOLO

# ---- class info (matches the dataset yaml in src/01) ----
NC = 9
CLASS_NAMES = {
    0: "Abrasion", 1: "Filling", 2: "Crown",
    3: "Caries 1 class", 4: "Caries 2 class", 5: "Caries 3 class",
    6: "Caries 4 class", 7: "Caries 5 class", 8: "Caries 6 class",
}

# ---- model dataset filenames (you upload these to /kaggle/input) ----
V6_MODEL_FILENAME = "V6_best.pt"
V13_MODEL_FILENAME = "V13_best.pt"

# Optional explicit overrides if auto-detection by filename fails.
MANUAL_V6_MODEL_PATH = None
MANUAL_V13_MODEL_PATH = None
MANUAL_VAL_IMAGES_DIR = None

# ---- tiling params: MUST match src/01 V13 training ----
TILE_SIZE = 640
OVERLAP = 0.20
MERGE_IOU = 0.50        # bbox-IoU threshold for merging detections across tiles

# ---- evaluation params ----
EVAL_CONF = 0.001       # LOW conf for mAP (NOT a submission threshold)
EVAL_MAX_DET = 300      # Ultralytics val default
IOU_THRES = 0.50        # within-tile / native NMS during predict
MASK_EVAL_LONG_SIDE = 640   # GT + preds rasterized to this long side for mask-IoU (SAME for both)
IOUV = np.linspace(0.5, 0.95, 10)   # 10 IoU thresholds, COCO-style

V6_IMG_SIZE = 768       # V6 was trained at imgsz=768 (native, no tiling)

# ---- memory / device ----
RETINA_MASKS = False
USE_HALF = True
DEVICE = 0 if torch.cuda.is_available() else "cpu"

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("TILE_SIZE:", TILE_SIZE, "| OVERLAP:", OVERLAP, "| MERGE_IOU:", MERGE_IOU)
print("EVAL_CONF:", EVAL_CONF, "| MAX_DET:", EVAL_MAX_DET, "| mask long side:", MASK_EVAL_LONG_SIDE)

## 3. Locate Validation Images + GT Labels

Looks for a `.../images/(valid|val)` directory that has a sibling `.../labels/(valid|val)`.

In [ ]:
def find_val_images_dir():
    if MANUAL_VAL_IMAGES_DIR is not None:
        return Path(MANUAL_VAL_IMAGES_DIR)
    candidates = []
    for images_dir in Path("/kaggle/input").rglob("images"):
        if not images_dir.is_dir():
            continue
        for sub in ("valid", "val"):
            vdir = images_dir / sub
            ldir = images_dir.parent / "labels" / sub
            if vdir.is_dir() and ldir.is_dir():
                candidates.append(vdir)
    if not candidates:
        raise FileNotFoundError(
            "Could not auto-locate a val images dir with a sibling labels dir. "
            "Set MANUAL_VAL_IMAGES_DIR."
        )
    return sorted(candidates,
                  key=lambda p: ("alpha" in str(p).lower(), "dent" in str(p).lower()),
                  reverse=True)[0]


VAL_IMAGES_DIR = find_val_images_dir()
LABELS_DIR = VAL_IMAGES_DIR.parent.parent / "labels" / VAL_IMAGES_DIR.name
val_image_paths = sorted([
    p for p in VAL_IMAGES_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
])

print("VAL_IMAGES_DIR  :", VAL_IMAGES_DIR)
print("LABELS_DIR      :", LABELS_DIR)
print("Val images      :", len(val_image_paths))
print("Label .txt files:", len(sorted(LABELS_DIR.glob("*.txt"))))
if len(val_image_paths) == 0:
    raise FileNotFoundError("No validation images found under VAL_IMAGES_DIR.")

## 4. Locate and Load the Two Checkpoints

By default this finds the files named `V6_best.pt` and `V13_best.pt` anywhere under
`/kaggle/input`.

In [ ]:
def locate_named_checkpoint(filename, manual=None):
    if manual is not None:
        p = Path(manual)
        if not p.exists():
            raise FileNotFoundError(f"Manual path does not exist: {p}")
        return p
    hits = sorted(Path("/kaggle/input").rglob(filename))
    if not hits:
        raise FileNotFoundError(
            f"Could not find {filename} under /kaggle/input. "
            f"Upload it as a Kaggle Dataset or set the manual override."
        )
    return hits[0]


V6_MODEL_PATH = locate_named_checkpoint(V6_MODEL_FILENAME, MANUAL_V6_MODEL_PATH)
V13_MODEL_PATH = locate_named_checkpoint(V13_MODEL_FILENAME, MANUAL_V13_MODEL_PATH)

print("V6_MODEL_PATH :", V6_MODEL_PATH)
print("V13_MODEL_PATH:", V13_MODEL_PATH)


def release_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


release_memory()
model_v6 = YOLO(str(V6_MODEL_PATH))
model_v13 = YOLO(str(V13_MODEL_PATH))
print("Loaded V6 and V13 models.")

## 5. Tiling Helpers

Inline copy of the geometry from `tools/tile_yolo_seg.py` (kept in sync). `compute_tiles` slices
an image, `untile_polygon` maps a tile-normalized polygon back to full-image coordinates, and
`merge_detections` de-duplicates the same object seen in overlapping tiles with class-wise NMS.

In [ ]:
def _starts(length, tile, step):
    if length <= tile:
        return [0]
    s = list(range(0, length - tile + 1, step))
    if s[-1] != length - tile:
        s.append(length - tile)
    return s


def compute_tiles(w, h, tile_size, overlap):
    """Pixel tile boxes (x0, y0, x1, y1) covering the whole image, edge tiles flush."""
    step = max(1, int(round(tile_size * (1.0 - overlap))))
    boxes = []
    for y0 in _starts(h, tile_size, step):
        for x0 in _starts(w, tile_size, step):
            boxes.append((x0, y0, min(x0 + tile_size, w), min(y0 + tile_size, h)))
    return boxes


def untile_polygon(poly_norm_tile, box, img_w, img_h):
    """Map a tile-normalized polygon back to a full-image-normalized polygon."""
    x0, y0, x1, y1 = box
    tw, th = x1 - x0, y1 - y0
    poly = np.asarray(poly_norm_tile, dtype=np.float64).reshape(-1, 2).copy()
    poly[:, 0] = (poly[:, 0] * tw + x0) / img_w
    poly[:, 1] = (poly[:, 1] * th + y0) / img_h
    return np.clip(poly, 0.0, 1.0)


def _poly_bbox(poly_norm):
    p = np.asarray(poly_norm, dtype=np.float64).reshape(-1, 2)
    return p[:, 0].min(), p[:, 1].min(), p[:, 0].max(), p[:, 1].max()


def _bbox_iou(a, b):
    ax0, ay0, ax1, ay1 = a
    bx0, by0, bx1, by1 = b
    ix0, iy0 = max(ax0, bx0), max(ay0, by0)
    ix1, iy1 = min(ax1, bx1), min(ay1, by1)
    iw, ih = max(0.0, ix1 - ix0), max(0.0, iy1 - iy0)
    inter = iw * ih
    if inter <= 0:
        return 0.0
    area_a = max(0.0, ax1 - ax0) * max(0.0, ay1 - ay0)
    area_b = max(0.0, bx1 - bx0) * max(0.0, by1 - by0)
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0


def merge_detections(detections, iou_thres=MERGE_IOU):
    """Class-wise greedy NMS over detections from all tiles of one image."""
    kept = []
    for det in sorted(detections, key=lambda d: d["confidence"], reverse=True):
        box = _poly_bbox(det["poly"])
        duplicate = False
        for k in kept:
            if k["class_id"] != det["class_id"]:
                continue
            if _bbox_iou(box, _poly_bbox(k["poly"])) > iou_thres:
                duplicate = True
                break
        if not duplicate:
            kept.append(det)
    return kept


def is_valid_polygon(polygon):
    if polygon is None:
        return False
    polygon = np.asarray(polygon)
    return polygon.ndim == 2 and polygon.shape[0] >= 3 and polygon.shape[1] == 2

## 6. Self-Contained Mask mAP (faithful to Ultralytics)

mask-IoU between predicted and GT masks, greedy same-class matching at each of 10 IoU thresholds,
then 101-point interpolated AP per class. Identical treatment for both models.

In [ ]:
def rasterize_poly(poly_norm, H, W):
    """Rasterize a full-image-normalized polygon [N,2] to a boolean mask [H,W]."""
    pts = np.round(np.asarray(poly_norm, dtype=np.float64) * np.array([W, H])).astype(np.int32)
    m = np.zeros((H, W), dtype=np.uint8)
    cv2.fillPoly(m, [pts.reshape(-1, 1, 2)], 1)
    return m.astype(bool)


def mask_iou_matrix(pred_masks, gt_masks):
    """pred_masks [Np, n] bool, gt_masks [Ng, n] bool -> IoU [Np, Ng]."""
    if pred_masks.shape[0] == 0 or gt_masks.shape[0] == 0:
        return np.zeros((pred_masks.shape[0], gt_masks.shape[0]), dtype=np.float32)
    pm = pred_masks.astype(np.float32)
    gm = gt_masks.astype(np.float32)
    inter = pm @ gm.T
    area_p = pm.sum(1)[:, None]
    area_g = gm.sum(1)[None, :]
    union = area_p + area_g - inter
    return inter / (union + 1e-7)


def match_predictions(pred_cls, gt_cls, iou, iouv):
    """Greedy IoU + same-class matching at each IoU threshold -> correct [Npred, niou] bool."""
    correct = np.zeros((pred_cls.shape[0], len(iouv)), dtype=bool)
    if iou.size == 0:
        return correct
    iou = iou.copy()
    iou[gt_cls[None, :] != pred_cls[:, None]] = 0.0   # only same-class matches count
    for i, thr in enumerate(iouv):
        matches = np.argwhere(iou >= thr)
        if matches.shape[0]:
            if matches.shape[0] > 1:
                order = iou[matches[:, 0], matches[:, 1]].argsort()[::-1]
                matches = matches[order]
                matches = matches[np.unique(matches[:, 1], return_index=True)[1]]  # one pred per gt
                matches = matches[np.unique(matches[:, 0], return_index=True)[1]]  # one gt per pred
            correct[matches[:, 0], i] = True
    return correct


# NumPy 2.0 renamed np.trapz -> np.trapezoid (and removed np.trapz). Resolve it once,
# without eagerly referencing the removed name (that would raise AttributeError).
_TRAPZ = np.trapezoid if hasattr(np, "trapezoid") else np.trapz


def compute_ap(recall, precision):
    """101-point interpolated AP (COCO / Ultralytics style)."""
    mrec = np.concatenate(([0.0], recall, [1.0]))
    mpre = np.concatenate(([1.0], precision, [0.0]))
    mpre = np.flip(np.maximum.accumulate(np.flip(mpre)))
    x = np.linspace(0, 1, 101)
    y = np.interp(x, mrec, mpre)
    return float(_TRAPZ(y, x))


def ap_per_class(tp, conf, pred_cls, target_cls, nc, iouv):
    """Returns ap [nc, niou]; classes absent from GT or never predicted stay 0."""
    niou = len(iouv)
    ap = np.zeros((nc, niou), dtype=np.float64)
    if tp.shape[0] == 0:
        return ap
    order = np.argsort(-conf)
    tp, pred_cls = tp[order], pred_cls[order]
    unique_classes, nt = np.unique(target_cls, return_counts=True)
    for c, n_l in zip(unique_classes, nt):
        c = int(c)
        i = pred_cls == c
        if i.sum() == 0 or n_l == 0:
            continue
        fpc = (1 - tp[i]).cumsum(0)
        tpc = tp[i].cumsum(0)
        recall = tpc / (n_l + 1e-16)
        precision = tpc / (tpc + fpc + 1e-16)
        for j in range(niou):
            ap[c, j] = compute_ap(recall[:, j], precision[:, j])
    return ap

## 7. Per-Model Inference + GT Loading + Evaluation Driver

In [ ]:
def predict_image_tiled(model, image_path):
    """V13-style: slice into tiles, predict each, untile polygons, merge across tiles."""
    img = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if img is None:
        return []
    h, w = img.shape[:2]
    detections = []
    for (x0, y0, x1, y1) in compute_tiles(w, h, TILE_SIZE, OVERLAP):
        crop = img[y0:y1, x0:x1]
        with torch.inference_mode():
            for result in model.predict(source=crop, imgsz=TILE_SIZE, conf=EVAL_CONF,
                                         iou=IOU_THRES, max_det=EVAL_MAX_DET, device=DEVICE,
                                         task="segment", retina_masks=RETINA_MASKS,
                                         half=USE_HALF if DEVICE != "cpu" else False,
                                         stream=True, verbose=False, save=False):
                if result.masks is None or result.boxes is None:
                    continue
                polys = result.masks.xyn
                cls = result.boxes.cls.cpu().numpy().astype(int)
                cf = result.boxes.conf.cpu().numpy().astype(float)
                for ci, cc, pp in zip(cls, cf, polys):
                    if not is_valid_polygon(pp):
                        continue
                    pf = untile_polygon(pp, (x0, y0, x1, y1), w, h)
                    if is_valid_polygon(pf):
                        detections.append({"class_id": int(ci), "confidence": float(cc), "poly": pf})
    merged = merge_detections(detections, iou_thres=MERGE_IOU)
    return [(d["class_id"], d["confidence"], d["poly"]) for d in merged]


def predict_image_native(model, image_path):
    """V6-style: native full-image inference, no tiling. masks.xyn are full-image normalized."""
    out = []
    with torch.inference_mode():
        for result in model.predict(source=str(image_path), imgsz=V6_IMG_SIZE, conf=EVAL_CONF,
                                     iou=IOU_THRES, max_det=EVAL_MAX_DET, device=DEVICE,
                                     task="segment", retina_masks=RETINA_MASKS,
                                     half=USE_HALF if DEVICE != "cpu" else False,
                                     stream=True, verbose=False, save=False):
            if result.masks is None or result.boxes is None:
                continue
            polys = result.masks.xyn
            cls = result.boxes.cls.cpu().numpy().astype(int)
            cf = result.boxes.conf.cpu().numpy().astype(float)
            for ci, cc, pp in zip(cls, cf, polys):
                if is_valid_polygon(pp):
                    out.append((int(ci), float(cc), np.asarray(pp, dtype=np.float64)))
    return out


def load_gt_polys(image_path):
    """Read YOLO-seg GT polygons (full-image normalized) for one image."""
    txt = LABELS_DIR / (Path(image_path).stem + ".txt")
    gts = []
    if not txt.exists():
        return gts
    for line in txt.read_text().strip().splitlines():
        parts = line.split()
        if len(parts) < 7:          # class id + at least 3 (x, y) points
            continue
        cls = int(float(parts[0]))
        coords = np.asarray(parts[1:], dtype=np.float64)
        coords = coords[: (len(coords) // 2) * 2].reshape(-1, 2)
        if coords.shape[0] >= 3:
            gts.append((cls, coords))
    return gts


def evaluate_model(predict_fn, model, image_paths):
    """Run inference over all val images and accumulate the raw matching results.

    Returns (tp, conf, pred_cls, target_cls) so the slow inference is decoupled from the fast
    AP computation: if the metric step needs a tweak/fix, you only re-run AP, not inference.
    """
    all_tp, all_conf, all_pcls, all_tcls = [], [], [], []
    for ip in tqdm(image_paths, desc="Eval"):
        img = cv2.imread(str(ip), cv2.IMREAD_COLOR)
        if img is None:
            continue
        h, w = img.shape[:2]
        scale = MASK_EVAL_LONG_SIDE / max(h, w)
        He, We = max(1, round(h * scale)), max(1, round(w * scale))

        preds = predict_fn(model, ip)
        gts = load_gt_polys(ip)

        pcls = np.array([p[0] for p in preds], dtype=int)
        pconf = np.array([p[1] for p in preds], dtype=float)
        gcls = np.array([g[0] for g in gts], dtype=int)

        pm = (np.stack([rasterize_poly(p[2], He, We) for p in preds]).reshape(len(preds), -1)
              if preds else np.zeros((0, He * We), dtype=bool))
        gm = (np.stack([rasterize_poly(g[1], He, We) for g in gts]).reshape(len(gts), -1)
              if gts else np.zeros((0, He * We), dtype=bool))

        iou = mask_iou_matrix(pm, gm)
        correct = match_predictions(pcls, gcls, iou, IOUV)

        all_tp.append(correct)
        all_conf.append(pconf)
        all_pcls.append(pcls)
        all_tcls.append(gcls)
        release_memory()

    tp = np.concatenate(all_tp, 0) if all_tp else np.zeros((0, len(IOUV)), dtype=bool)
    conf = np.concatenate(all_conf, 0) if all_conf else np.zeros((0,), dtype=float)
    pcls = np.concatenate(all_pcls, 0) if all_pcls else np.zeros((0,), dtype=int)
    tcls = np.concatenate(all_tcls, 0) if all_tcls else np.zeros((0,), dtype=int)
    return tp, conf, pcls, tcls

## 8. Run Both Evaluations (identical metric)

In [ ]:
print("=== Evaluating V13 (tiled + merged) on full val images ===")
tp_v13, conf_v13, pcls_v13, tcls = evaluate_model(predict_image_tiled, model_v13, val_image_paths)

print(f"=== Evaluating V6 baseline (native full image, imgsz={V6_IMG_SIZE}) ===")
tp_v6, conf_v6, pcls_v6, tcls_v6 = evaluate_model(predict_image_native, model_v6, val_image_paths)

# GT is the same set for both runs; sanity check the label set matched.
assert np.array_equal(np.sort(tcls), np.sort(tcls_v6)), "GT class sets differ between runs."
print("Inference done. Raw arrays cached in the kernel; AP is computed in the next cell")
print("(so re-running the metric does NOT re-run the slow inference).")

## 9. Results: V13 vs V6, Same Metric

`Delta` is the number that decides V13. The historical baseline is ~0.234; compare against the V6
row re-scored here (same code), not against 0.234 directly.

In [ ]:
# AP is computed here from the cached raw arrays (cell above), so a tweak/fix in the metric
# does NOT require re-running the slow inference.
ap_v13 = ap_per_class(tp_v13, conf_v13, pcls_v13, tcls, NC, IOUV)
ap_v6 = ap_per_class(tp_v6, conf_v6, pcls_v6, tcls_v6, NC, IOUV)

gt_classes = np.unique(tcls).astype(int)


def summarize(ap, tag):
    m5095 = float(ap[gt_classes].mean()) if len(gt_classes) else 0.0
    m50 = float(ap[gt_classes, 0].mean()) if len(gt_classes) else 0.0
    print(f"{tag}: Mask mAP50 = {m50:.4f} | Mask mAP50-95 = {m5095:.4f}")
    return m50, m5095


print("Metric: full-image mask mAP, IDENTICAL code for both models (fair V13-vs-V6 delta).")
print("History baseline reference: ~0.234 Mask mAP50-95 (V6/V10).\n")

v13_50, v13_5095 = summarize(ap_v13, "V13 tiled ")
v6_50,  v6_5095  = summarize(ap_v6,  "V6 native ")
print(f"\nDelta (V13 - V6) Mask mAP50-95 = {v13_5095 - v6_5095:+.4f}")

rows = []
for c in gt_classes:
    c = int(c)
    rows.append({
        "class_id": c,
        "name": CLASS_NAMES.get(c, str(c)),
        "n_gt": int((tcls == c).sum()),
        "V13_mAP50-95": round(float(ap_v13[c].mean()), 4),
        "V6_mAP50-95": round(float(ap_v6[c].mean()), 4),
    })
per_class = pd.DataFrame(rows)
per_class["delta"] = (per_class["V13_mAP50-95"] - per_class["V6_mAP50-95"]).round(4)
display(per_class)